# 05 - Conclusion & Discussion

---

## 1. Problem Introduction

This project addresses the problem of **hotel booking cancellation prediction**, a practical and important challenge in the hospitality industry.

**Problem statement:** Predict whether a booking will be canceled using information available at the time of booking.

The target variable is `is_canceled`, where:

- `0` = Not canceled
- `1` = Canceled

**Business objective:** Support hotel management teams in:

- Identifying bookings with high cancellation risk early
- Making more informed overbooking decisions
- Optimizing revenue and operational resource allocation
- Designing better guest retention strategies

---

## 2. Dataset Description

The dataset used in this project is the **Hotel Booking Demand** dataset from Kaggle.

- **Dataset size:** 119,390 bookings and 32 original columns
- **Time period:** July 2015 to August 2017
- **Hotel types:** City Hotel and Resort Hotel
- **Target variable:** `is_canceled`

The target variable shows whether each booking was canceled or not:

- Not canceled: 75,166 bookings
- Canceled: 44,224 bookings

This means that approximately 37% of bookings in the original dataset were canceled.

Some important features in the dataset include:

- `hotel`: type of hotel, either City Hotel or Resort Hotel
- `lead_time`: number of days between booking date and arrival date
- `deposit_type`: deposit policy of the booking
- `adr`: average daily rate per night
- `market_segment`: booking market segment
- `customer_type`: type of customer
- `previous_cancellations`: number of previous cancellations
- `total_of_special_requests`: number of special requests made by the guest

---

## 3. Data Cleaning Summary

Before preprocessing and model training, the original dataset was cleaned to improve data quality and reduce modeling issues.

| Step | Action | Outcome |
|---|---|---|
| Missing values | Filled `children`, `agent`, and `company` with 0; filled `country` with `"Unknown"` | No missing values remained |
| Duplicated rows | Checked exact duplicated rows but did not remove them | Duplicate issue documented as a limitation |
| Text standardization | Removed leading and trailing spaces from text columns | More consistent categorical values |
| Undefined categories | Replaced `meal = Undefined` with `SC`; replaced undefined booking channels with `Unknown` | Clearer categorical values |
| Date variables | Created `arrival_date` and converted date columns into datetime format | Date information became easier to use |
| Feature engineering | Created `total_guests`, `total_nights`, `has_agent`, and `has_company` | New meaningful variables added |
| Invalid records | Removed bookings with 0 guests or 0 nights | Logically invalid records removed |
| ADR values | Removed negative ADR and ADR values above the 99th percentile | Cleaner price distribution |
| Lead time | Checked high lead time values but did not remove them | Preserved valid long-term bookings |
| Guest values | Checked unusual guest-related values but did not remove them automatically | Avoided removing possible group or family bookings |
| Data leakage | Dropped `reservation_status` and `reservation_status_date` | Prevented the model from learning the answer directly |

**Important note:** Exact duplicated rows were retained because the dataset does not contain a unique booking ID or customer ID. Therefore, identical rows may represent either true duplicated records or separate bookings with the same characteristics.

This is a known limitation of the dataset and may affect the reliability of model evaluation.

---

## 4. Data Preprocessing Summary

After cleaning, the dataset was prepared for machine learning.
The preprocessing stage included:
- Separating the target variable `is_canceled` from the feature variables
- Removing columns not suitable for direct modeling
- Identifying numerical and categorical features
- Splitting the dataset into training and testing sets
- Applying `StandardScaler` to numerical variables
- Applying `OneHotEncoder` to categorical variables
- Creating a Scikit-Learn preprocessing pipeline
The dataset was split using an 80/20 train-test split:
- 80% for training
- 20% for testing
The split was performed using a time-based approach: the dataset was sorted by `arrival_date`, and the first 80% of records (chronologically) were used for training while the remaining 20% were used for testing.
This approach helps avoid data leakage from future bookings into the training set and better simulates real-world deployment conditions.
The preprocessing pipeline was fitted only on the training data to prevent data leakage.

---

## 5. Key Insights from the Analysis

### Insight 1 — Deposit type is strongly related to cancellation

`deposit_type` is one of the most important features in predicting cancellations.

Bookings with different deposit policies show very different cancellation behaviors. This suggests that deposit policy is not only a payment condition, but also a behavioral signal.

In particular, some deposit types may be strongly associated with specific booking channels or customer behaviors.

### Insight 2 — Longer lead time is associated with higher cancellation risk

`lead_time` represents the number of days between booking date and arrival date.

A longer lead time often means that the guest booked far in advance. These bookings may have a higher cancellation risk because travel plans can change over time.

This feature is useful because it is known at the time of booking, making it practical for early cancellation risk prediction.

### Insight 3 — City Hotel and Resort Hotel show different cancellation patterns

The two hotel types have different booking behaviors.

City Hotel bookings tend to be more related to short stays, business trips, and flexible travel plans. Resort Hotel bookings may be more connected to leisure travel and planned vacations.

This difference makes `hotel` an important feature for cancellation prediction.

### Insight 4 — Special requests may indicate stronger booking intention

The feature `total_of_special_requests` can reflect guest engagement.

Guests who make special requests may have already thought more carefully about their stay. Therefore, they may be less likely to cancel compared to guests who make no special requests.

### Insight 5 — Previous cancellation behavior is useful

`previous_cancellations` provides information about past guest behavior.

Guests who have canceled bookings before may have a higher probability of canceling again. This makes previous cancellation history a useful predictor in the model.

---

## 6. Model Results

Two models were trained and evaluated to predict hotel booking cancellations.

The dataset was split using a **time-based 80/20 split** — training on earlier
bookings and testing on the most recent period (April 2017 – August 2017) —
to simulate a realistic forecasting scenario.

- **Train set:** 93,916 bookings
- **Test set:** 23,480 bookings (arrival period: 2017-04-20 → 2017-08-31)

---

### 6.1 Performance Comparison

| Metric | Logistic Regression (Baseline) | XGBoost (Main Model) |
|---|---|---|
| **Accuracy** | 76.99% | **81.40%** |
| **F1-Score (macro)** | 0.7640 | **0.8037** |
| F1 — Not Canceled (class 0) | 0.8003 | **0.8470** |
| F1 — Canceled (class 1) | 0.7305 | **0.7615** |
| **ROC-AUC** | 0.8573 | **0.9077** |
| Precision — Not Canceled | 0.8159 | **0.8169** |
| Precision — Canceled | 0.7093 | **0.8069** |
| Recall — Not Canceled | 0.7852 | **0.8794** |
| Recall — Canceled | **0.7473** | 0.7189 |

XGBoost outperforms Logistic Regression on every primary metric except Recall
for the Canceled class, where Logistic Regression is marginally higher (0.7473
vs 0.7189).

---

### 6.2 Confusion Matrix — XGBoost (Main Model)

| | Predicted: Not Canceled | Predicted: Canceled |
|---|---|---|
| **Actual: Not Canceled** | 12,136 ✅ | 1,665 ❌ |
| **Actual: Canceled** | 2,721 ❌ | 6,958 ✅ |

- **True Negatives — 12,136:** guests predicted to stay who actually stayed
- **True Positives — 6,958:** cancellations correctly identified
- **False Positives — 1,665:** flagged as high-risk but guest actually showed up
- **False Negatives — 2,721:** missed cancellations — rooms expected to be
  occupied that will actually be empty

---

### 6.3 XGBoost Top 15 Feature Importances

| Rank | Feature | Importance Score |
|---|---|---|
| 1 | deposit_type_No Deposit | 0.5239 |
| 2 | deposit_type_Non Refund | 0.1760 |
| 3 | market_segment_Online TA | 0.0334 |
| 4 | deposit_type_Refundable | 0.0256 |
| 5 | required_car_parking_spaces | 0.0251 |
| 6 | previous_cancellations | 0.0173 |
| 7 | country_PRT | 0.0109 |
| 8 | market_segment_Offline TA/TO | 0.0086 |
| 9 | total_of_special_requests | 0.0083 |
| 10 | customer_type_Transient | 0.0075 |
| 11 | country_ITA | 0.0061 |
| 12 | reserved_room_type_A | 0.0055 |
| 13 | country_DEU | 0.0047 |
| 14 | arrival_date_year | 0.0045 |
| 15 | country_AGO | 0.0043 |

`deposit_type` dominates the model — the three deposit type features together account for a large share of the total feature importance score. This is consistent with EDA: Non Refund bookings show very high cancellation rates, while No Deposit bookings have a lower cancellation rate. This makes deposit policy one of the most informative variables available at booking time.

---

### 6.4 Assessment

- **XGBoost achieves ROC-AUC = 0.9077** — a strong result indicating the model reliably separates canceled from non-canceled bookings across all probability thresholds.
- **XGBoost improves over the Logistic Regression baseline across all primary
metrics:** +4.36% accuracy, +0.0397 F1 macro, +0.0497 ROC-AUC
- **Precision for Canceled class jumps from 0.7093 to 0.8069** — XGBoost
  generates significantly fewer false alarms, meaning hotel teams can act on
  high-risk flags with greater confidence
- **Recall for Canceled class is slightly lower in XGBoost (0.7189 vs 0.7473)**
  — the model catches fewer cancellations in total but is more precise about
  the ones it flags; this is an acceptable trade-off for operational use cases
  where alert fatigue is a concern
- **False Negatives (2,721)** remain the costliest error type — each represents
  a room expected to be occupied that will actually be empty, directly impacting
  revenue. Reducing false negatives should be the primary focus of future
  model improvements
- Feature importance results strongly validate EDA findings: `deposit_type`,
  `market_segment`, `previous_cancellations`, and `total_of_special_requests`
  are important in both the exploratory analysis and the model's learned behavior
- The time-based split means the test set covers a different time window than
  the training set, providing a more realistic estimate of real-world
  generalization than a random split would

---

## 7. Practical Recommendations

Based on the analysis and model results, several practical recommendations can be made for hotel management teams.

### 1. Use early-warning cancellation risk flags

Hotels can use the prediction model to identify bookings with high cancellation risk soon after they are created.

Bookings with high-risk patterns, such as long lead time, certain deposit types, or specific booking channels, can be flagged for closer monitoring.

### 2. Improve communication with high-risk guests

For bookings predicted to have a high cancellation probability, hotels can send follow-up messages to confirm guest intention.

Examples include:

- Confirmation emails
- Personalized reminders
- Stay preview messages
- Upgrade or flexible check-in offers

This can help reduce uncertainty and encourage guests to keep their bookings.

### 3. Apply smarter overbooking decisions

Hotels can use predicted cancellation risk to support overbooking decisions.

Instead of using one general cancellation rate, hotel managers can use risk-based information to make more careful decisions, especially during high-demand periods.

### 4. Pay attention to lead time

Bookings made very far in advance should be monitored carefully.

Long lead time does not mean the booking is invalid, but it may indicate greater uncertainty. Hotels can use this information to design reminder strategies before arrival.

### 5. Prioritize guests with special requests

Guests with special requests may show stronger booking intention.

Hotels can treat these bookings as higher-intent reservations and prioritize them in guest service planning.

---

## 8. Limitations

This project has several limitations.

### 1. Duplicate rows were not removed

The dataset contains many exact duplicated rows. However, because there is no unique booking ID or customer ID, it is not possible to confirm whether these rows are true duplicates or different bookings with identical characteristics.

As a result, duplicated rows were kept. This decision preserves possible real bookings but may also affect model evaluation.

### 2. Limited feature engineering

Only basic feature engineering was applied, such as:

- `total_guests`
- `total_nights`
- `has_agent`
- `has_company`
- `arrival_date`

More advanced features, such as seasonality, holidays, or interaction terms, were not fully explored.

### 3. Dataset time period is limited

The dataset covers bookings from 2015 to 2017.

Booking behavior may have changed over time, especially after the COVID-19 pandemic. Therefore, the model should be retrained on more recent data before real-world deployment.

### 4. Business context is limited

The dataset does not include some useful business information, such as:

- Booking ID
- Customer ID
- Exact booking timestamp
- Room revenue after cancellation
- Customer communication history

These missing details limit the depth of business interpretation.

---

## 9. Future Improvements

Future work could further improve both the predictive performance and practical applicability of this project in several ways.

### 1. Expand and Compare More Machine Learning Models

This project currently evaluates both Logistic Regression and XGBoost models. Future research could extend the comparison by including additional machine learning algorithms such as:

- Random Forest
- LightGBM
- CatBoost
- Support Vector Machine (SVM)

Comparing multiple models would provide a deeper understanding of model strengths, interpretability, computational efficiency, and predictive capability. Ensemble and boosting methods may further improve performance by capturing complex nonlinear relationships within booking behavior.

---

### 2. Improve Feature Engineering

Additional feature engineering could help the models capture more meaningful customer and booking patterns. Potential new features include:

- Arrival season
- Arrival quarter
- Weekend stay indicator
- Family booking indicator
- Booking modification frequency
- Interaction between `lead_time` and `deposit_type`
- Historical cancellation rate by market segment
- Customer loyalty indicators

These engineered features may improve the model’s ability to detect subtle cancellation patterns and increase overall prediction accuracy. 

---

## 10. Final Conclusion

This project shows that hotel booking cancellations can be predicted using booking information available before the final booking outcome.
After data cleaning, preprocessing, and Logistic Regression and XGBoost model training, the project demonstrates that several features are useful for cancellation prediction, especially deposit policy, lead time, hotel type, previous cancellation behavior, and special requests.
The model can support hotel managers by identifying high-risk bookings earlier, improving guest communication, supporting overbooking decisions, and helping reduce revenue loss from cancellations.

This project uses Logistic Regression as the baseline model and XGBoost as the main model, providing a stronger foundation for future improvement with richer feature engineering and better business integration.
